# 🚀 End-to-End RAG Agent Deployment with Gradio & Streamlit

This notebook walks through building and deploying a **Retrieval-Augmented Generation (RAG)** agent from scratch, with two UI options:
- **Gradio** – quick, shareable demo interface
- **Streamlit** – rich, production-ready web app

---
## 🗂️ Workflow Overview
```
1. Install Dependencies
2. Load & Chunk Documents
3. Build Vector Store (FAISS)
4. Create RAG Chain (LangChain + OpenAI)
5. Deploy with Gradio
6. Deploy with Streamlit
7. (Optional) HuggingFace / Local LLM variant
```

## 📦 Step 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q \
    langchain \
    langchain-openai \
    langchain-community \
    faiss-cpu \
    sentence-transformers \
    gradio \
    streamlit \
    pypdf \
    python-dotenv \
    tiktoken \
    openai

print("✅ All packages installed!")

## 🔑 Step 2 — Configure API Keys

In [ ]:
import os
from dotenv import load_dotenv

# Option A: Load from .env file
load_dotenv()

# Option B: Set directly (replace with your actual key)
# os.environ["OPENAI_API_KEY"] = "sk-..."

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY_HERE")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("✅ API key configured" if OPENAI_API_KEY != "YOUR_OPENAI_API_KEY_HERE" else "⚠️  Please set your OPENAI_API_KEY")

## 📄 Step 3 — Load & Prepare Documents

Supports: **PDF, plain text, or any custom corpus**

In [ ]:
from langchain.document_loaders import PyPDFLoader, TextLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Option A: Load a PDF ──────────────────────────────────────────────────────
# loader = PyPDFLoader("your_document.pdf")

# ── Option B: Load all PDFs from a folder ────────────────────────────────────
# loader = DirectoryLoader("./docs", glob="**/*.pdf", loader_cls=PyPDFLoader)

# ── Option C: Use sample in-memory text (demo) ───────────────────────────────
from langchain.schema import Document

sample_texts = [
    """Artificial Intelligence (AI) refers to simulation of human intelligence in machines.
    Machine learning is a subset of AI that enables systems to learn from data automatically.
    Deep learning uses neural networks with many layers to model complex patterns in data.""",

    """Retrieval-Augmented Generation (RAG) is an AI technique that enhances LLM responses
    by fetching relevant documents from a knowledge base before generating an answer.
    RAG reduces hallucinations and grounds responses in factual, up-to-date information.""",

    """LangChain is a framework for building applications powered by language models.
    It provides tools for chaining prompts, memory, agents, and retrieval systems.
    FAISS (Facebook AI Similarity Search) is a fast vector similarity search library.""",

    """Gradio is an open-source Python library for building ML demo UIs instantly.
    Streamlit is a Python framework for creating interactive data science web apps.
    Both tools allow rapid prototyping and sharing of AI applications.""",

    """Vector databases store embeddings — numerical representations of text.
    When a query is made, it is embedded and compared to stored vectors using cosine similarity.
    Top-k most similar chunks are retrieved and passed to the LLM as context."""
]

raw_documents = [Document(page_content=text, metadata={"source": f"sample_{i}"}) 
                 for i, text in enumerate(sample_texts)]

print(f"📄 Loaded {len(raw_documents)} document(s)")

# ── Chunk the documents ──────────────────────────────────────────────────────
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
)

chunks = text_splitter.split_documents(raw_documents)
print(f"✂️  Split into {len(chunks)} chunks")
print(f"\n📝 Sample chunk:\n{chunks[0].page_content}")

## 🧠 Step 4 — Build Vector Store with FAISS

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

print("⏳ Creating embeddings and building FAISS index...")

# Initialize embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Build vector store
vectorstore = FAISS.from_documents(chunks, embeddings)

# Save index to disk for reuse
vectorstore.save_local("faiss_index")

print("✅ FAISS vector store built and saved to ./faiss_index")
print(f"📦 Total vectors stored: {vectorstore.index.ntotal}")

## 🔗 Step 5 — Build the RAG Chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# ── Custom RAG Prompt Template ────────────────────────────────────────────────
RAG_PROMPT = """
You are a helpful AI assistant. Use the following retrieved context to answer 
the user's question accurately and concisely.

If the context does not contain enough information to answer, say:
"I don't have enough information in my knowledge base to answer that."

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=RAG_PROMPT
)

# ── Initialize LLM ────────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0.2,
    max_tokens=512
)

# ── Build Retriever ───────────────────────────────────────────────────────────
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Retrieve top-3 chunks
)

# ── Assemble RAG Chain ────────────────────────────────────────────────────────
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

print("✅ RAG chain assembled and ready!")

## 🧪 Step 6 — Test the RAG Agent (CLI)

In [ ]:
def query_rag(question: str) -> dict:
    """Query the RAG agent and return answer + sources."""
    result = rag_chain.invoke({"query": question})
    answer = result["result"]
    sources = [doc.metadata.get("source", "unknown") for doc in result["source_documents"]]
    return {"answer": answer, "sources": list(set(sources))}


# ── Quick Tests ───────────────────────────────────────────────────────────────
test_questions = [
    "What is RAG and why is it useful?",
    "What is LangChain used for?",
    "How does FAISS help in AI applications?",
]

for q in test_questions:
    response = query_rag(q)
    print(f"\n❓ Question: {q}")
    print(f"💬 Answer: {response['answer']}")
    print(f"📚 Sources: {response['sources']}")
    print("-" * 70)

## 🎨 Step 7 — Deploy with Gradio

Gradio spins up a shareable demo UI — great for quick prototyping and demos.
Running inside Jupyter creates an inline interface.

In [ ]:
import gradio as gr

# ── Chat History State ────────────────────────────────────────────────────────
chat_history = []

def rag_chat(user_message: str, history: list):
    """Gradio chat function with RAG backend."""
    if not user_message.strip():
        return history, ""
    
    # Query RAG chain
    response = query_rag(user_message)
    answer = response["answer"]
    sources = ", ".join(response["sources"]) if response["sources"] else "N/A"
    
    # Format response with source attribution
    formatted = f"{answer}\n\n📚 *Sources: {sources}*"
    history.append((user_message, formatted))
    return history, ""


def clear_chat():
    return [], ""


# ── Build Gradio UI ───────────────────────────────────────────────────────────
with gr.Blocks(
    title="RAG Agent",
    theme=gr.themes.Soft(primary_hue="indigo"),
    css=".gradio-container { max-width: 860px !important; margin: auto; }"
) as gradio_app:
    
    gr.Markdown("""
    # 🤖 RAG Agent — Gradio Interface
    Ask questions about your knowledge base. The agent retrieves relevant context before answering.
    """)
    
    with gr.Row():
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(
                label="Conversation",
                height=420,
                bubble_full_width=False
            )
            
            with gr.Row():
                txt_input = gr.Textbox(
                    placeholder="Ask something about your documents...",
                    label="Your Question",
                    scale=5,
                    lines=1
                )
                send_btn = gr.Button("Send 🚀", variant="primary", scale=1)
            
            with gr.Row():
                clear_btn = gr.Button("🗑️ Clear Chat", variant="secondary")
        
        with gr.Column(scale=1):
            gr.Markdown("### 💡 Example Questions")
            examples = gr.Examples(
                examples=[
                    ["What is RAG?"],
                    ["How does FAISS work?"],
                    ["What is LangChain?"],
                    ["Explain deep learning"],
                ],
                inputs=txt_input,
                label=""
            )
            
            gr.Markdown("### ⚙️ Model Info")
            gr.Markdown("""
            - **LLM:** GPT-4o-mini  
            - **Embeddings:** text-embedding-3-small  
            - **Vector DB:** FAISS  
            - **Top-K chunks:** 3  
            """)
    
    # ── Wire up events ────────────────────────────────────────────────────────
    send_btn.click(
        fn=rag_chat,
        inputs=[txt_input, chatbot],
        outputs=[chatbot, txt_input]
    )
    txt_input.submit(
        fn=rag_chat,
        inputs=[txt_input, chatbot],
        outputs=[chatbot, txt_input]
    )
    clear_btn.click(fn=clear_chat, outputs=[chatbot, txt_input])


# ── Launch ────────────────────────────────────────────────────────────────────
# share=True creates a public URL (valid for 72h)
gradio_app.launch(share=False, inline=True, debug=False)
print("\n🌐 Gradio app running! Set share=True for a public URL.")

## 🖥️ Step 8 — Deploy with Streamlit

The cell below writes a complete **`streamlit_app.py`** file. 
Run it externally with: `streamlit run streamlit_app.py`

In [ ]:
streamlit_code = '''
# streamlit_app.py — RAG Agent with Streamlit
# Run with: streamlit run streamlit_app.py

import os
import streamlit as st
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Page Config ───────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="RAG Agent",
    page_icon="🤖",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ── Custom CSS ────────────────────────────────────────────────────────────────
st.markdown("""
<style>
    .stChatMessage { border-radius: 12px; padding: 4px; }
    .block-container { padding-top: 2rem; }
    .metric-card { background: #f0f2f6; border-radius: 8px; padding: 12px; }
</style>
""", unsafe_allow_html=True)

# ── Sidebar ───────────────────────────────────────────────────────────────────
with st.sidebar:
    st.image("https://via.placeholder.com/200x60?text=RAG+Agent", width=200)
    st.title("⚙️ Configuration")
    
    api_key = st.text_input(
        "OpenAI API Key", 
        type="password",
        value=os.getenv("OPENAI_API_KEY", ""),
        help="Enter your OpenAI API key"
    )
    
    model_name = st.selectbox(
        "LLM Model",
        ["gpt-4o-mini", "gpt-4o", "gpt-3.5-turbo"],
        index=0
    )
    
    top_k = st.slider("Top-K Retrieval Chunks", 1, 10, 3)
    temperature = st.slider("LLM Temperature", 0.0, 1.0, 0.2, 0.1)
    
    st.divider()
    st.markdown("### 📊 Session Stats")
    if "messages" in st.session_state:
        total_qs = sum(1 for m in st.session_state.messages if m["role"] == "user")
        st.metric("Questions Asked", total_qs)
    
    if st.button("🗑️ Clear Chat", use_container_width=True):
        st.session_state.messages = []
        st.rerun()

# ── Main Header ───────────────────────────────────────────────────────────────
st.title("🤖 RAG Agent — Streamlit Interface")
st.caption("Retrieval-Augmented Generation powered by LangChain + FAISS + OpenAI")

# ── Load / Build Vector Store (cached) ───────────────────────────────────────
@st.cache_resource
def build_rag_chain(api_key: str, model: str, top_k: int, temperature: float):
    """Build and cache the RAG chain."""
    os.environ["OPENAI_API_KEY"] = api_key
    
    # Sample corpus (replace with your data loader)
    sample_texts = [
        "Artificial Intelligence (AI) refers to simulation of human intelligence in machines. "
        "Machine learning is a subset of AI that enables systems to learn from data automatically.",
        "Retrieval-Augmented Generation (RAG) enhances LLM responses by fetching relevant "
        "documents from a knowledge base before generating an answer, reducing hallucinations.",
        "LangChain is a framework for building applications powered by language models. "
        "FAISS is a fast vector similarity search library used for efficient retrieval.",
        "Gradio and Streamlit are Python tools for building interactive AI demo applications quickly.",
        "Vector databases store embeddings and retrieve top-k similar chunks via cosine similarity."
    ]
    
    docs = [Document(page_content=t, metadata={"source": f"doc_{i}"}) 
            for i, t in enumerate(sample_texts)]
    
    splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
    chunks = splitter.split_documents(docs)
    
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})
    
    prompt = PromptTemplate(
        input_variables=["context", "question"],
        template="""
        You are a helpful AI assistant. Use the context below to answer accurately.
        If insufficient context, say so.

        Context: {context}
        Question: {question}
        Answer:"""
    )
    
    llm = ChatOpenAI(model_name=model, temperature=temperature, max_tokens=512)
    
    chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={"prompt": prompt}
    )
    return chain


# ── Initialize Chat State ─────────────────────────────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages = [{
        "role": "assistant",
        "content": "Hi! I\'m your RAG agent. Ask me anything about the knowledge base! 📚"
    }]

# ── Display Chat History ──────────────────────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if "sources" in msg:
            with st.expander("📚 Retrieved Sources"):
                for src in msg["sources"]:
                    st.code(src)

# ── Chat Input ────────────────────────────────────────────────────────────────
if user_input := st.chat_input("Ask a question about your documents..."):
    
    if not api_key:
        st.error("⚠️ Please enter your OpenAI API key in the sidebar.")
        st.stop()
    
    # Add user message
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)
    
    # Generate RAG response
    with st.chat_message("assistant"):
        with st.spinner("🔍 Retrieving context and generating answer..."):
            try:
                rag_chain = build_rag_chain(api_key, model_name, top_k, temperature)
                result = rag_chain.invoke({"query": user_input})
                answer = result["result"]
                sources = [doc.page_content[:120] + "..." 
                           for doc in result["source_documents"]]
                
                st.markdown(answer)
                with st.expander("📚 Retrieved Sources"):
                    for i, src in enumerate(sources, 1):
                        st.markdown(f"**Chunk {i}:** {src}")
                
                st.session_state.messages.append({
                    "role": "assistant",
                    "content": answer,
                    "sources": sources
                })
            except Exception as e:
                st.error(f"❌ Error: {str(e)}")
'''

with open("streamlit_app.py", "w") as f:
    f.write(streamlit_code)

print("✅ streamlit_app.py written!")
print("\n🚀 To launch Streamlit, run in terminal:")
print("   streamlit run streamlit_app.py")

## 🔁 Step 9 — Launch Streamlit from Notebook

You can launch Streamlit as a background process from within the notebook:

In [ ]:
import subprocess, sys, time, threading

def run_streamlit():
    subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", "streamlit_app.py",
         "--server.port", "8501", "--server.headless", "true"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
time.sleep(3)

print("🌐 Streamlit app launched!")
print("   Open in browser: http://localhost:8501")
print("\n💡 For Google Colab, use ngrok or localtunnel to expose the port.")

## 🤗 Step 10 — (Optional) Local HuggingFace Variant

Use a **free, offline LLM** instead of OpenAI — no API key needed.

In [ ]:
# Uncomment to install HuggingFace dependencies
# !pip install -q transformers torch accelerate sentencepiece

# from langchain_community.llms import HuggingFacePipeline
# from langchain_community.embeddings import HuggingFaceEmbeddings
# from transformers import pipeline

# # Use local embeddings (no OpenAI needed)
# hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# vectorstore_hf = FAISS.from_documents(chunks, hf_embeddings)

# # Use a local LLM (e.g., flan-t5-base for fast testing)
# pipe = pipeline("text2text-generation", model="google/flan-t5-base", max_length=512)
# llm_hf = HuggingFacePipeline(pipeline=pipe)

# retriever_hf = vectorstore_hf.as_retriever(search_kwargs={"k": 3})
# rag_chain_hf = RetrievalQA.from_chain_type(
#     llm=llm_hf,
#     retriever=retriever_hf,
#     return_source_documents=True
# )
# result = rag_chain_hf.invoke({"query": "What is RAG?"})
# print(result["result"])

print("ℹ️  Uncomment the code above to use a local HuggingFace model (no API key required).")

## ☁️ Step 11 — Deploy to Cloud

### Gradio — Deploy to HuggingFace Spaces
```bash
# Create a new Space at huggingface.co/new-space
# Select SDK: Gradio
# Push your code:
git init && git add . && git commit -m "RAG agent"
git remote add origin https://huggingface.co/spaces/YOUR_USERNAME/rag-agent
git push origin main
```

### Streamlit — Deploy to Streamlit Cloud
```bash
# 1. Push your code to GitHub
# 2. Go to share.streamlit.io
# 3. Connect repo → select streamlit_app.py → Deploy!
# 4. Add OPENAI_API_KEY in Secrets settings
```

### Required files for deployment
```
requirements.txt  ← list all pip packages
.env              ← (local only, add to .gitignore)
faiss_index/      ← pre-built index (or build at startup)
```

In [ ]:
# Generate requirements.txt for deployment
requirements = """langchain>=0.2.0
langchain-openai>=0.1.0
langchain-community>=0.2.0
faiss-cpu>=1.7.4
sentence-transformers>=2.2.2
gradio>=4.0.0
streamlit>=1.32.0
pypdf>=4.0.0
python-dotenv>=1.0.0
tiktoken>=0.5.0
openai>=1.0.0
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("✅ requirements.txt created for deployment!")
print(requirements)

---
## ✅ Summary

| Step | Component | Description |
|------|-----------|-------------|
| 1 | Dependencies | LangChain, FAISS, Gradio, Streamlit, OpenAI |
| 2 | API Keys | `.env` or direct config |
| 3 | Document Loading | PDF / text / directory |
| 4 | Text Chunking | `RecursiveCharacterTextSplitter` |
| 5 | Embeddings | `text-embedding-3-small` (OpenAI) |
| 6 | Vector Store | FAISS (local, persisted to disk) |
| 7 | RAG Chain | `RetrievalQA` + custom prompt |
| 8 | Gradio UI | Chat interface with source display |
| 9 | Streamlit UI | Full-featured app with sidebar controls |
| 10 | Local LLM | HuggingFace variant (no API key) |
| 11 | Deployment | HF Spaces (Gradio) / Streamlit Cloud |

---
### 📁 Output Files
- `streamlit_app.py` — full Streamlit app
- `requirements.txt` — pip dependencies
- `faiss_index/` — persisted vector store

---
*Built with LangChain • FAISS • OpenAI • Gradio • Streamlit*